# fyM1 — vocabulary trim (plan Task 7)

Drops tokenizer entries, and their embedding / output rows, that Malagasy + French + English text never
uses. The trimmed model ships **only if all three gates pass**; otherwise the numbers are reported as a
finding and the untrimmed GGUF ships.

- **(a) round trip** — 2,000 held-out Malagasy lines decode to the same text with the trimmed tokenizer.
- **(b) same answers** — greedy answers to the 150 eval prompts are identical to the untrimmed model's for ≥ 95 % of prompts.
- **(c) Belebele `plt_Latn`** — accuracy within 1 point of the untrimmed model (same scorer both sides).
- **Export check** — the trimmed GGUF tokenizes held-out lines like the trimmed HF tokenizer, at least as often as the untrimmed GGUF matches the untrimmed HF tokenizer.

Gates (b) and (c) compare the merged 16-bit checkpoint with its trimmed copy, both loaded the same way
(4-bit), so they test exactly what gets exported.

**Runtime:** T4 GPU. **Secret:** `HF_TOKEN` (write access to `Fan1000-Prog/fyM1-ckpt`). Repo code comes
from `fyM1-ckpt/code/` (see `docs/colab_operator.md`); it must hold `src/fym1/shrink/*.py` and
`eval/prompts.jsonl` from `main`.
**Outputs:** `fyM1-ckpt/results/shrink.json` (counts and scores only; copy it to `results/shrink.json` in the
repo). On pass: `fyM1-ckpt/trimmed/` (trimmed Q4_K_M GGUF + `keep_ids.json`).

**Privacy:** this notebook reads the anonymised private chats. It prints counts only. Never print chat lines,
and clear outputs before saving the notebook anywhere.

In [ ]:
# Parameters — set BASE_MODEL_ID from results/model_selection.md before running.
BASE_MODEL_ID = ''                      # e.g. 'Qwen/Qwen3-8B'; must be the model the adapter was trained on
CKPT_REPO = 'Fan1000-Prog/fyM1-ckpt'    # private: LoRA adapter in, results + trimmed GGUF out
ADAPTER_SUBFOLDER = 'final'
CHATS_REPO = 'Fan1000-Prog/fyM1-chats'  # private dataset: chats.jsonl, formal.jsonl, identity.jsonl
SYSTEM_PROMPT = ''                      # the exact system prompt used in 01_train.ipynb
CHAT_TEMPLATE_KWARGS = {'enable_thinking': False}  # same chat-template options as training and serving

HELD_OUT_LINES = 2000                   # gate (a)
FR_EN_LINES = 50_000                    # half French, half English Wikipedia lines
GREEDY_MATCH_MIN = 0.95                 # gate (b)
BELEBELE_MAX_DROP = 1.0                 # gate (c), accuracy points
MAX_NEW_TOKENS = 256
GEN_BATCH = 16
BELEBELE_BATCH = 8
SEED = 42
LLAMA_CPP_PIN = '9d1878a4354e743529fbe09e7350a957f5e39e9a'  # tested; the GGUF step relies on its converter internals
COMPUTE_DTYPE = 'float16'               # use 'float32' for Gemma 3 on T4 (fp16 activations overflow)

WORK = '/content/trim'
assert BASE_MODEL_ID, 'set BASE_MODEL_ID'
assert SYSTEM_PROMPT, 'set SYSTEM_PROMPT (same as training)'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1. Install and log in

In [ ]:
import os
import sys
from pathlib import Path

from google.colab import userdata
from huggingface_hub import snapshot_download

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # never print it
CODE = Path(snapshot_download(CKPT_REPO, allow_patterns=['code/*'])) / 'code'
missing = [p for p in ('src/fym1/shrink/vocab.py', 'src/fym1/shrink/model.py', 'eval/prompts.jsonl') if not (CODE / p).exists()]
assert not missing, f'upload these repo files to {CKPT_REPO}/code/ first: {missing}'
sys.path.insert(0, str(CODE / 'src'))

!git init -q /content/llama.cpp && git -C /content/llama.cpp fetch -q --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP_PIN} && git -C /content/llama.cpp checkout -q FETCH_HEAD
!pip install -q unsloth
!pip install -q sentencepiece protobuf safetensors datasets -e /content/llama.cpp/gguf-py
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
# llama-quantize is the only llama.cpp binary needed (GGUF f16 -> Q4_K_M). CPU build, ~5 min.
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
LLAMA_CPP_COMMIT = !git -C /content/llama.cpp rev-parse HEAD
LLAMA_CPP_COMMIT = LLAMA_CPP_COMMIT[0]
print('llama.cpp', LLAMA_CPP_COMMIT)

## 2. Merge the LoRA adapter into a 16-bit checkpoint

Unsloth merges shard by shard, so this fits in Colab RAM. Unsloth must be imported before transformers.

In [ ]:
import gc
import json
import os
import random
from pathlib import Path

import unsloth  # noqa: F401  (must come first)
import torch
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

os.makedirs(WORK, exist_ok=True)
MERGED = Path(WORK) / 'merged16'
TRIMMED = Path(WORK) / 'trimmed16'

adapter_dir = Path(snapshot_download(CKPT_REPO, allow_patterns=[f'{ADAPTER_SUBFOLDER}/*'])) / ADAPTER_SUBFOLDER
adapter_base = json.loads((adapter_dir / 'adapter_config.json').read_text())['base_model_name_or_path']
print('adapter trained on:', adapter_base)
if BASE_MODEL_ID.split('/')[-1].lower() not in adapter_base.lower():
    print(f'WARNING: adapter base {adapter_base!r} does not look like BASE_MODEL_ID {BASE_MODEL_ID!r}')

model, tokenizer = FastLanguageModel.from_pretrained(str(adapter_dir), max_seq_length=2048, load_in_4bit=True)
model.save_pretrained_merged(str(MERGED), tokenizer, save_method='merged_16bit')
del model, tokenizer
gc.collect(); torch.cuda.empty_cache()
print('merged:', sorted(p.name for p in MERGED.iterdir()))

## 3. Corpus → token ids to keep

Keep corpus = every training text (chats, formal slice, identity lines) rendered with the chat template
and system prompt, plus each prefix rendered with the generation prompt (the header the model answers after), minus the held-out lines, plus French and English Wikipedia lines. The 150 eval prompts
and Belebele are **not** in the corpus, so gates (b) and (c) also test unseen text.

In [ ]:
import itertools

from datasets import load_dataset
from transformers import AutoTokenizer

from fym1.shrink.vocab import chat_texts, tokens_to_keep, trim_tokenizer

tokenizer = AutoTokenizer.from_pretrained(MERGED)

chats_dir = Path(snapshot_download(CHATS_REPO, repo_type='dataset', allow_patterns=['*.jsonl']))
conversations = []
for path in sorted(chats_dir.glob('**/*.jsonl')):
    for line in path.read_text(encoding='utf-8').splitlines():
        record = json.loads(line)
        messages = record['messages'] if isinstance(record, dict) else record
        conversations.append([m for m in messages if m.get('content')])
print('conversations:', len(conversations))

rng = random.Random(SEED)
rng.shuffle(conversations)
held_out_lines, train_convs = [], []
for conv in conversations:  # hold out whole conversations so no held-out line is in the corpus
    if len(held_out_lines) < HELD_OUT_LINES:
        held_out_lines += [m['content'] for m in conv]
    else:
        train_convs.append(conv)
held_out_lines = held_out_lines[:HELD_OUT_LINES]
assert len(held_out_lines) == HELD_OUT_LINES, len(held_out_lines)


def wiki_lines(config, n):
    rows = load_dataset('wikimedia/wikipedia', config, split='train', streaming=True)
    lines = (l for row in rows for l in row['text'].split('\n') if 20 <= len(l) <= 1000)
    return list(itertools.islice(lines, n))


# Whole conversations plus every generation-prompt prefix: the header the model answers after only appears there.
corpus = list(chat_texts(tokenizer, train_convs, SYSTEM_PROMPT, **CHAT_TEMPLATE_KWARGS))
fr_en = wiki_lines('20231101.fr', FR_EN_LINES // 2) + wiki_lines('20231101.en', FR_EN_LINES // 2)
keep = tokens_to_keep(tokenizer, corpus + fr_en, always_keep=set())
print(f'corpus: {len(train_convs)} conversations ({len(corpus)} rendered texts) + {len(fr_en)} fr/en lines; held out: {len(held_out_lines)} lines')
print(f'vocab {len(tokenizer)} -> keep {len(keep)} ({len(keep) / len(tokenizer):.1%})')

## 4. Gate (a): tokenizer round trip on held-out lines

In [ ]:
new_tokenizer = trim_tokenizer(tokenizer, keep)


def ids(tok, text):
    return tok.encode(text, add_special_tokens=False)


equal = sum(new_tokenizer.decode(ids(new_tokenizer, l)) == tokenizer.decode(ids(tokenizer, l)) for l in held_out_lines)
token_ratio = sum(len(ids(new_tokenizer, l)) for l in held_out_lines) / sum(len(ids(tokenizer, l)) for l in held_out_lines)
corpus_same = sum([keep[i] for i in ids(new_tokenizer, t)] == ids(tokenizer, t) for t in corpus)
gate_a = {
    'lines': len(held_out_lines), 'decode_equal': equal, 'token_count_ratio': round(token_ratio, 4),
    'corpus_same_ids': corpus_same, 'corpus_size': len(corpus), 'pass': equal == len(held_out_lines),
}
print(gate_a)

## 5. Trim the checkpoint

Shard by shard: only the embedding and output matrices change; config and generation config ids are
renumbered; `tokenizer.model` is trimmed too when present (llama.cpp reads it for SentencePiece models).

In [ ]:
from safetensors import safe_open

from fym1.shrink.model import trim_checkpoint

trim_checkpoint(MERGED, TRIMMED, keep)


def param_count(directory):
    total = 0
    for shard in Path(directory).glob('*.safetensors'):
        with safe_open(shard, framework='pt') as f:
            for name in f.keys():
                n = 1
                for d in f.get_slice(name).get_shape():
                    n *= d
                total += n
    return total


params = {'before': param_count(MERGED), 'after': param_count(TRIMMED)}
params['saved_share'] = round(1 - params['after'] / params['before'], 4)
print(params)

## 6. Gates (b) and (c): same answers, same Belebele

Both checkpoints load in 4-bit with plain transformers (same settings), one at a time.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

DTYPE = getattr(torch, COMPUTE_DTYPE)
QUANT = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=DTYPE)
prompts = [json.loads(l) for l in (CODE / 'eval/prompts.jsonl').read_text(encoding='utf-8').splitlines()]
belebele = load_dataset('facebook/belebele', 'plt_Latn', split='test')
print('prompts:', len(prompts), 'belebele:', len(belebele))


def load(directory):
    tok = AutoTokenizer.from_pretrained(directory)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(directory, quantization_config=QUANT, device_map='cuda:0', torch_dtype=DTYPE)
    return mdl.eval(), tok


@torch.no_grad()
def greedy_answers(mdl, tok):
    texts = [tok.apply_chat_template([{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': p['prompt']}],
                                     tokenize=False, add_generation_prompt=True, **CHAT_TEMPLATE_KWARGS) for p in prompts]
    answers = []
    for start in range(0, len(texts), GEN_BATCH):
        batch = tok(texts[start:start + GEN_BATCH], return_tensors='pt', padding=True, add_special_tokens=False).to(mdl.device)
        out = mdl.generate(**batch, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
        answers += tok.batch_decode(out[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
    return answers


def belebele_prompt(row):
    options = '\n'.join(f'{letter}. {row[f"mc_answer{i}"]}' for i, letter in enumerate('ABCD', start=1))
    return f"{row['flores_passage']}\n\nQuestion: {row['question']}\n{options}\nAnswer:"


@torch.no_grad()
def belebele_accuracy(mdl, tok):
    letters = [ids(tok, f' {letter}') for letter in 'ABCD']
    assert all(len(l) == 1 for l in letters), letters
    letters = [l[0] for l in letters]
    correct = 0
    for start in range(0, len(belebele), BELEBELE_BATCH):
        rows = belebele.select(range(start, min(start + BELEBELE_BATCH, len(belebele))))
        batch = tok([belebele_prompt(r) for r in rows], return_tensors='pt', padding=True).to(mdl.device)
        logits = mdl(**batch).logits[:, -1, letters]
        predicted = logits.argmax(-1).tolist()
        correct += sum(p == int(r['correct_answer_num']) - 1 for p, r in zip(predicted, rows))
    return 100 * correct / len(belebele)


runs = {}
for name, directory in [('untrimmed', MERGED), ('trimmed', TRIMMED)]:
    mdl, tok = load(directory)
    runs[name] = {'answers': greedy_answers(mdl, tok), 'belebele': belebele_accuracy(mdl, tok)}
    print(name, 'belebele', round(runs[name]['belebele'], 2))
    del mdl
    gc.collect(); torch.cuda.empty_cache()

# Identical garbage would pass gate (b): require the untrimmed model to beat chance and to answer.
assert runs['untrimmed']['belebele'] > 30, 'untrimmed Belebele near chance (25%): check COMPUTE_DTYPE before trusting gates'
assert sum(bool(a.strip()) for a in runs['untrimmed']['answers']) >= 0.9 * len(prompts), 'untrimmed model gave empty answers'
identical = sum(a == b for a, b in zip(runs['untrimmed']['answers'], runs['trimmed']['answers']))
gate_b = {'prompts': len(prompts), 'identical': identical, 'rate': round(identical / len(prompts), 4),
          'min': GREEDY_MATCH_MIN, 'pass': identical / len(prompts) >= GREEDY_MATCH_MIN}
drop = runs['untrimmed']['belebele'] - runs['trimmed']['belebele']
gate_c = {'before': round(runs['untrimmed']['belebele'], 2), 'after': round(runs['trimmed']['belebele'], 2),
          'max_drop': BELEBELE_MAX_DROP, 'pass': drop <= BELEBELE_MAX_DROP}
print(gate_b)
print(gate_c)

## 7. GGUF export, size and speed (both models, same llama.cpp build)

llama.cpp identifies byte-level BPE pre-tokenizers by hashing the token ids of a fixed test string. Trimming
renumbers ids, so the hash no longer matches; we patch the lookup to return the name the **untrimmed**
tokenizer resolves to (same regex, same merges order). SentencePiece models (Gemma) are read from
`tokenizer.model` and do not use the hash.
The patch only affects conversion: the GGUF gets the standard `tokenizer.ggml.pre` value (e.g. `qwen2`), so stock
llama.cpp, llama-cpp-python and Ollama load it unchanged.

In [ ]:
import runpy
import subprocess
import sys
import time

sys.path[:0] = ['/content/llama.cpp', '/content/llama.cpp/gguf-py']
import conversion.base as gguf_base
import llama_cpp

try:
    PRE_TOKENIZER = gguf_base.TextModel.get_vocab_base_pre(None, AutoTokenizer.from_pretrained(MERGED))
except NotImplementedError:
    PRE_TOKENIZER = None  # not a recognised byte-level BPE; only matters if the converter takes the gpt2 path
print('untrimmed pre-tokenizer:', PRE_TOKENIZER)


def to_gguf(directory, name, pre):
    f16 = Path(WORK) / f'{name}-f16.gguf'
    q4 = Path(WORK) / f'{name}-Q4_K_M.gguf'
    original = gguf_base.TextModel.get_vocab_base_pre
    if pre is not None:
        gguf_base.TextModel.get_vocab_base_pre = lambda self, tok: pre
    sys.argv = ['convert_hf_to_gguf.py', str(directory), '--outfile', str(f16), '--outtype', 'f16']
    try:
        runpy.run_path('/content/llama.cpp/convert_hf_to_gguf.py', run_name='__main__')
    finally:
        gguf_base.TextModel.get_vocab_base_pre = original
    subprocess.run(['/content/llama.cpp/build/bin/llama-quantize', str(f16), str(q4), 'Q4_K_M'], check=True,
                   stdout=subprocess.DEVNULL)
    f16.unlink()
    return q4


def gguf_tokenizer_agreement(q4, tok):
    llm = llama_cpp.Llama(model_path=str(q4), vocab_only=True, verbose=False)
    same = sum(llm.tokenize(l.encode(), add_bos=False, special=False) == ids(tok, l) for l in held_out_lines)
    return same / len(held_out_lines)


def cpu_tokens_per_second(q4, n_prompts=3, max_tokens=64):
    llm = llama_cpp.Llama(model_path=str(q4), n_ctx=1024, n_threads=2, verbose=False)  # free HF Space: 2 vCPU
    produced, start = 0, time.time()
    for p in prompts[:n_prompts]:
        out = llm.create_chat_completion([{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': p['prompt']}],
                                         max_tokens=max_tokens, temperature=0)
        produced += out['usage']['completion_tokens']
    return produced / (time.time() - start)


gguf = {}
for name, directory, tok in [('untrimmed', MERGED, tokenizer), ('trimmed', TRIMMED, new_tokenizer)]:
    q4 = to_gguf(directory, name, PRE_TOKENIZER if name == 'trimmed' else None)
    gguf[name] = {'file': q4.name, 'bytes': q4.stat().st_size,
                  'tokenizer_agreement': round(gguf_tokenizer_agreement(q4, tok), 4),
                  'cpu_tokens_per_s_2_threads': round(cpu_tokens_per_second(q4), 2)}
    print(name, gguf[name])
gguf['size_ratio'] = round(gguf['trimmed']['bytes'] / gguf['untrimmed']['bytes'], 4)
gguf['pass'] = gguf['trimmed']['tokenizer_agreement'] >= gguf['untrimmed']['tokenizer_agreement']

## 8. Verdict and results

In [ ]:
import datetime
import importlib.metadata as metadata

from huggingface_hub import HfApi

passed = all(g['pass'] for g in (gate_a, gate_b, gate_c, gguf))
results = {
    # Flat keys read by paper/make_numbers.py; the nested blocks below keep the per-gate detail.
    'gates_passed': passed,
    'vocab_before': len(tokenizer), 'vocab_after': len(keep),
    'params_before': params['before'], 'params_after': params['after'],
    'gguf_bytes_before': gguf['untrimmed']['bytes'], 'gguf_bytes_after': gguf['trimmed']['bytes'],
    'tokens_per_sec_before': gguf['untrimmed']['cpu_tokens_per_s_2_threads'],
    'tokens_per_sec_after': gguf['trimmed']['cpu_tokens_per_s_2_threads'],
    'base_model': BASE_MODEL_ID,
    'adapter': f'{CKPT_REPO}/{ADAPTER_SUBFOLDER}',
    'tokens_per_sec_hardware': 'Colab CPU, llama-cpp-python, 2 threads, Q4_K_M',
    'corpus': {'conversations': len(train_convs), 'rendered_texts': len(corpus), 'fr_en_lines': len(fr_en)},
    'params': params,
    'gates': {'a_round_trip': gate_a, 'b_greedy_identical': gate_b, 'c_belebele_plt_Latn': gate_c, 'gguf_export': gguf},
    'versions': {'llama.cpp': LLAMA_CPP_COMMIT, **{p: metadata.version(p) for p in ('transformers', 'tokenizers', 'torch', 'bitsandbytes', 'unsloth', 'llama_cpp_python')}},
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
print(json.dumps(results, indent=2))
print('VERDICT:', 'SHIP TRIMMED' if passed else 'DO NOT SHIP TRIMMED (report as finding)')

api = HfApi()
results_path = Path(WORK) / 'shrink.json'
results_path.write_text(json.dumps(results, indent=2))
api.upload_file(path_or_fileobj=str(results_path), path_in_repo='results/shrink.json', repo_id=CKPT_REPO,
                commit_message='Vocabulary trim results')
if passed:
    keep_path = Path(WORK) / 'keep_ids.json'
    keep_path.write_text(json.dumps(keep))
    api.upload_file(path_or_fileobj=str(keep_path), path_in_repo='trimmed/keep_ids.json', repo_id=CKPT_REPO,
                    commit_message='Trimmed vocabulary ids')
    api.upload_file(path_or_fileobj=str(Path(WORK) / gguf['trimmed']['file']), path_in_repo=f"trimmed/{gguf['trimmed']['file']}",
                    repo_id=CKPT_REPO, commit_message='Trimmed Q4_K_M GGUF')
print('uploaded')